In [1]:
cd ..

/workspace/llm-graph-construction


/opt/conda/lib/python3.10/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [2]:
from training.train_and_evaluate_relation_extraction import *

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


/workspace/llm-graph-construction


In [3]:
test_name = "i2b2"
_, _, dataset_test = load_stored_dataset_combination_graph(balanced=False, dataset=test_name)
model = torch.load("evaluation_results/bimodal-model-" + test_name + ".pt")

In [4]:
dataset_test.generated[0]

Data(x=[77, 768], edge_index=[2, 304], edge_attr=[304, 775], y=[1], edge_type=[304], event1_index=[1], event2_index=[1], text_relations=[264], text='A few days later she complained of <e1>dizziness</e1> .
This was mostly described as feeling unsteady on her feet .
She stated that she had trouble knowing if her feet were touching the ground .
She was seen again by primary care physician , Roderick since she had complaints of <e2>dizziness</e2> , as well', event1_start=39, event1_end=48, event2_start=283, event2_end=292)

In [35]:
# Select text of interest
# print(set(dataset_test.df["document_id"]))
document_id = "342"
start = 1310
end = 1516
df = dataset_test.df
df = df[df["document_id"] == document_id]
df = df[df["event1_start"] >= start]
df = df[df["event2_start"] >= start]
df = df[df["event1_end"] <= end]
df = df[df["event2_end"] <= end]
besedilo = df[df["document_id"] == document_id].iloc[0]["text"][start:end]
df

,text,class,event1_start,event1_end,event1_type,event2_start,event2_end,event2_type,event1_text,event2_text,document_id,source,additional_document_info,minutes_between_means,event1_start_time,event2_start_time,event1_end_time,event2_end_time
550,\nAdmission Date :\n2013-07-23\nDischarge Date...,AFTER,1353,1362,None,1317,1348,None,intubated,increasing respiratory distress,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
551,\nAdmission Date :\n2013-07-23\nDischarge Date...,AFTER,1375,1399,None,1353,1362,None,conventional ventilation,intubated,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
552,\nAdmission Date :\n2013-07-23\nDischarge Date...,OVERLAP,1404,1413,None,1375,1399,None,pressures,conventional ventilation,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
553,\nAdmission Date :\n2013-07-23\nDischarge Date...,OVERLAP,1424,1428,None,1375,1399,None,rate,conventional ventilation,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
554,\nAdmission Date :\n2013-07-23\nDischarge Date...,AFTER,1462,1470,None,1353,1362,None,Survanta,intubated,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
556,\nAdmission Date :\n2013-07-23\nDischarge Date...,OVERLAP,1495,1514,None,1375,1399,None,ventilator settings,conventional ventilation,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
557,\nAdmission Date :\n2013-07-23\nDischarge Date...,OVERLAP,1480,1486,None,1495,1514,None,weaned,ventilator settings,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN
558,\nAdmission Date :\n2013-07-23\nDischarge Date...,OVERLAP,1487,1491,None,1480,1486,None,well,weaned,342,I2B2,"{'times': [('ADMISSION', '2013-07-23'), ('DISC...",NaN,NaN,NaN,NaN,NaN


In [41]:
def get_events(df, start_offset):
    events = []
    event_pairs = []
    for i, row in df.iterrows():
        e1 = (row["event1_start"] - start_offset, row["event1_end"] - start_offset, row["event1_text"])
        e2 = (row["event2_start"] - start_offset, row["event2_end"] - start_offset, row["event2_text"])
        events.append(e1)
        events.append(e2)
        event_pairs.append((e1, e2))
    
        
    return list(set(events)), event_pairs

In [44]:
# Prepare knowledge graphs
from pipeline.pipeline import *
patient_id = 0
events, event_pairs = get_events(df, start)
dataframe = construct_basic_dataframe(besedilo, event_pairs, 0)
dataset = construct_dataset_with_graphs(besedilo, dataframe, patient_id)
relations = predict_temporal_relations(dataset)

https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectors_sparse.npz not found in cache, downloading to /tmp/tmp_vbwxyn8
Finished download, copying /tmp/tmp_vbwxyn8 to cache at /root/.scispacy/datasets/2b79923846fb52e62d686f2db846392575c8eb5b732d9d26cd3ca9378c622d40.87bd52d0f0ee055c1e455ef54ba45149d188552f07991b765da256a1b512ca0b.tfidf_vectors_sparse.npz
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/nmslib_index.bin not found in cache, downloading to /tmp/tmpemkobkrp
Finished download, copying /tmp/tmpemkobkrp to cache at /root/.scispacy/datasets/7e8e091ec80370b87b1652f461eae9d926e543a403a69c1f0968f71157322c25.6d801a1e14867953e36258b0e19a23723ae84b0abd2a723bdd3574c3e0c873b4.nmslib_index.bin
https://s3-us-west-2.amazonaws.com/ai2-s2-scispacy/data/linkers/2023-04-23/umls/tfidf_vectorizer.joblib not found in cache, downloading to /tmp/tmp3qy8o7jm
Finished download, copying /tmp/tmp3qy8o7jm to cache at /root/.scispacy/da

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)



[===============================================---] 95.8% 63.2/66.0MB downloaded

IOPub message rate exceeded.
The Jupyter server will temporarily stop sending output
to the client in order to avoid crashing it.
To change this limit, set the config variable
`--ServerApp.iopub_msg_rate_limit`.

Current values:
ServerApp.iopub_msg_rate_limit=1000.0 (msgs/sec)
ServerApp.rate_limit_window=3.0 (secs)

Downloading: 100%|███████████████████████████| 226k/226k [00:00<00:00, 1.12MB/s]
Downloading: 100%|████████████████████████████| 48.0/48.0 [00:00<00:00, 132kB/s]
Downloading: 100%|█████████████████████████████| 570/570 [00:00<00:00, 1.60MB/s]
Downloading: 100%|███████████████████████████| 420M/420M [00:05<00:00, 74.7MB/s]
Some weights of the model checkpoint at bert-base-uncased were not used when initializing BertModel: ['cls.predictions.transform.LayerNorm.bias', 'cls.seq_relationship.bias', 'cls.predictions.decoder.weight', 'cls.predictions.bias', 'cls.predictions.transform.LayerNorm.weight', 'cls.seq_relationship.weight', 'cls.predictions.transform.dense.bias', 'cls.pre

Accuracy: 0.4642857142857143
Number of relations: 1.0

 ... executing workers ...

['oxygen', 'is a', 'requirement during', 'intubation']
['complications', 'of', 'intubation', 'include', 'pneumothorax']
['length of stay', 'is affected by']
Graph 0 generated
['oxygen', 'is a', 'requirement during', 'intubation']
['complications', 'of', 'intubation', 'include', 'pneumothorax']
Graph 1 generated
Graph 2 generated
Graph 3 generated
['oxygen', 'is a', 'requirement during', 'intubation']
['complications', 'of', 'intubation', 'include', 'pneumothorax']
Graph 4 generated
Graph 5 generated
Graph 6 generated
Graph 7 generated
Generated 8 examples


In [47]:
correct_relations = []
for i, row in df.iterrows():
    correct_relations.append((row["event1_text"],row["class"], row["event2_text"]))

In [45]:
relations

[('increasing respiratory distress', 'AFTER', 'intubated'),
 ('intubated', 'BEFORE', 'conventional ventilation'),
 ('conventional ventilation', 'OVERLAP', 'pressures'),
 ('conventional ventilation', 'BEFORE', 'rate'),
 ('intubated', 'BEFORE', 'Survanta'),
 ('conventional ventilation', 'BEFORE', 'ventilator settings'),
 ('weaned', 'BEFORE', 'ventilator settings'),
 ('weaned', 'BEFORE', 'well')]

In [48]:
correct_relations

[('intubated', 'AFTER', 'increasing respiratory distress'),
 ('conventional ventilation', 'AFTER', 'intubated'),
 ('pressures', 'OVERLAP', 'conventional ventilation'),
 ('rate', 'OVERLAP', 'conventional ventilation'),
 ('Survanta', 'AFTER', 'intubated'),
 ('ventilator settings', 'OVERLAP', 'conventional ventilation'),
 ('weaned', 'OVERLAP', 'ventilator settings'),
 ('well', 'OVERLAP', 'weaned')]

# Old attempts

In [38]:
# Filter generated
rows_to_consider = set()
for i, row in df.iterrows():
    rows_to_consider.add((row["event1_text"].strip(), row["event2_text"].strip()))

new_generated = []
for graph in dataset_test.generated:
    event1_text = graph["text"][graph["event1_start"]:graph["event1_end"]]
    event2_text = graph["text"][graph["event2_start"]:graph["event2_end"]]
    if (event1_text.strip(), event2_text.strip()) in rows_to_consider:
        print(event1_text, event2_text)
        new_generated.append(graph)
new_generated

[]

In [28]:
dataset_test.generated = list(filter(lambda x: x is not None, map(window_text, new_generated)))
dataLoader = DataLoader(dataset_test, batch_size=1)

In [29]:
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)

MultiModalPrediction(
  (graph_model): GraphEncoder(
    (criterion): CrossEntropyLoss()
    (softmax): Softmax(dim=1)
    (convs): ModuleList(
      (0-1): 2 x TemporalRelationAggregation()
    )
    (lns): ModuleList(
      (0): LayerNorm((50,), eps=1e-05, elementwise_affine=True)
    )
    (linear): Linear(in_features=50, out_features=50, bias=True)
    (post_mp): Sequential(
      (0): Linear(in_features=100, out_features=50, bias=True)
      (1): Dropout(p=0.2, inplace=False)
      (2): LeakyReLU(negative_slope=0.01)
      (3): Linear(in_features=50, out_features=3, bias=True)
    )
  )
  (text_model): EntityBERTtextEncoder(
    (EntityBert): BertModel(
      (embeddings): BertEmbeddings(
        (word_embeddings): Embedding(30539, 768, padding_idx=0)
        (position_embeddings): Embedding(512, 768)
        (token_type_embeddings): Embedding(2, 768)
        (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
        (dropout): Dropout(p=0.1, inplace=False)
      )

In [30]:
relation_types = ["BEFORE", "AFTER", "OVERLAP"]
predictions = []
correct_graph = []
for graph in dataLoader:
    graph.to(device)
    prediction = model(graph, graph.y)
    # predictions = np.argmax(logits, axis=-1)
    cls = np.argmax(prediction["predictions"].cpu().detach().numpy(), axis=-1)
    for i in range(len(graph["text"])):
        event1 = graph["text"][i][graph["event1_start"][i]:graph["event1_end"][i]]
        event2 = graph["text"][i][graph["event2_start"][i]:graph["event2_end"][i]]
        predictions.append((event1, relation_types[cls[i]], event2))
        correct_graph.append((event1, relation_types[graph.y[i]], event2))

In [31]:
predictions

[]

In [46]:
correct_graph

[]

In [33]:
besedilo

'Due to increasing respiratory distress was intubated , placed on conventional ventilation and pressures of 24/6 , rate of 25 .\nShe received one dose of Survanta and then weaned well on ventilator settings .'

In [34]:
df[df["document_id"] == document_id].iloc[0]["text"]

"\nAdmission Date :\n2013-07-23\nDischarge Date :\n2013-08-01\nService :\nCMED CSRU\nHISTORY :\nBaby Girl Kathryn Frazier , triplet III , delivered at 32 - 5/7 weeks gestation was admitted to the newborn intensive care unit for management of prematurity .\nBirth weight was 1630 grams .\nThe mother is a 31 - year-old Gravida 5 , para 1 , now 4 woman with estimated date of delivery 2013-09-12 .\nPrenatal screens included blood type A+ , antibody screen negative , rubella immune , RPR nonreactive .\nHepatitis B surface antigen negative , cystic fibrosis negative , and Group B strep unknown .\nMother 's medical history was notable for depression treated with Zoloft .\nOB history notable for infertility treated with Clomid .\nThis pregnancy was complicate by triplet gestation , cervical shortening , and pregnancy induced hypertension .\nDelivery was by cesarean section under spinal anesthesia for pre-eclampsia .\nThere was no labor or maternal fever .\nMembranes were ruptured at delivery fo